In [1]:
%cd ../../..

/home/hoanghu/projects/Food-Waste-Optimization


In [2]:
from functools import reduce


import numpy as np
import pandas as pd
import polars as pl
from pyspark.sql.session import SparkSession
import pyspark.sql.functions as pF
from pyspark.sql import DataFrame

In [3]:
spark = (
    SparkSession.builder.master("local[*]")
    .appName("YLVA")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    # .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4")
    # .config("spark.hadoop.fs.s3a.access.key", os.environ.get("AWS_ACCESS_KEY_ID"))
    # .config(
    #     "spark.hadoop.fs.s3a.secret.key", os.environ.get("AWS_SECRET_ACCESS_KEY")
    # )
    .config("spark.driver.memory", f"{4}g")
    .config("spark.executor.memory", f"{8}g")
    .config("spark.dynamicAllocation.enabled", "true")
    .config("spark.shuffle.service.enabled", "true")
    .getOrCreate()
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/11 14:52:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Read data

In [4]:
path = "data/processed/phase_4/dim_meals.xlsx"
dim_meals_raw = spark.createDataFrame(pd.read_excel(path))

dim_meals_raw.show()

+-------+-----------+----------+-------+------+----------+-------------------+
|meal_id|meal_type_1|schoolyear|is_kela|is_new|restaurant|        meal_type_2|
+-------+-----------+----------+-------+------+----------+-------------------+
|   9017|      vegan|     24-25|   true|  true|   che-exa|vegan-miscellaneous|
|   7201|      vegan|     23-24|  false| false|      NULL|               NULL|
|   9032|      vegan|     23-24|  false| false|      NULL|               NULL|
|   9102|      vegan|     23-24|  false| false|      NULL|               NULL|
|   7010| vegetarian|     24-25|  false| false|   che-exa|               NULL|
|   7100|      vegan|     23-24|  false| false|      NULL|               NULL|
|   5265|    chicken|     23-24|  false| false|      NULL|               NULL|
|   6557|      vegan|     24-25|   true|  true|   che-exa|    today’s special|
|   1751|    chicken|     24-25|   true|  true|   che-exa|               NULL|
|   9039|      vegan|     24-25|  false| false|   ch

# Generate menu candidates

In [5]:
NUM_VEGAN_PER_DAY = 2
NUM_MEALS_PER_DAY = [3, 4]
NUM_COMBOS_PER_DAY = 10
MAX_MEAL_OCCURENCES = 2

In [6]:
restaurant = "che"
schoolyear = "24-25"

In [7]:
date = "2024-12-02"

In [8]:
meals = (
    dim_meals_raw
    .filter(
        (dim_meals_raw.restaurant.isNotNull())
        & (dim_meals_raw.restaurant.contains(restaurant))
        & (dim_meals_raw.schoolyear == schoolyear)
    )
    .withColumn("id", pF.monotonically_increasing_id())
)

## 1. Generate day-level menus

In [9]:
# Compose vegan combo
meals_vegan = meals.filter(meals.meal_type_1 == "vegan").select("id", "meal_id")
combo_vegan = (
    meals_vegan
    .select(
        pF.col('meal_id').alias('meal_id_1'),
        pF.col('id').alias('id_1')
    )
    .crossJoin(meals_vegan)
    .filter(pF.col("id_1") < pF.col("id"))
    .select(
        pF.col("meal_id_1").alias("vegan_1"),
        pF.col("meal_id").alias("vegan_2"),
    )
)



# Compose non-vegan combo
meals_notvegan = meals.filter(meals.meal_type_1 != "vegan").select("id", "meal_id")

meals_notvegan_single = meals_notvegan.select(
    pF.col('meal_id').alias("nonvegan_1"),
    pF.lit(-1).alias("nonvegan_2"),
)
meals_notvegan_multi = (
    meals_notvegan
    .select(
        pF.col('meal_id').alias('meal_id_1'),
        pF.col('id').alias('id_1')
    )
    .crossJoin(meals_notvegan)
    .filter(pF.col("id_1") < pF.col("id"))
    .select(
        pF.col("meal_id_1").alias("nonvegan_1"),
        pF.col("meal_id").alias("nonvegan_2"),
    )
)
combo_notvegan = reduce(DataFrame.unionAll, [meals_notvegan_single, meals_notvegan_multi])



# Compose combo
is_kela = meals.select("meal_id", "is_kela")

menus_day = (
    combo_vegan
    .crossJoin(combo_notvegan)
    .join(
        meals.select(
            pF.col("meal_id").alias("vegan_1"),
            pF.when(pF.col("is_kela"), 1).otherwise(0).alias("kela_vegan_1")
        ),
        on='vegan_1',
        how='left'
    )
    .join(
        meals.select(
            pF.col("meal_id").alias("vegan_2"),
            pF.when(pF.col("is_kela"), 1).otherwise(0).alias("kela_vegan_2")
        ),
        on='vegan_2',
        how='left'
    )
    .join(
        meals.select(
            pF.col("meal_id").alias("nonvegan_1"),
            pF.when(pF.col("is_kela"), 1).otherwise(0).alias("kela_nonvegan_1")
        ),
        on='nonvegan_1',
        how='left'
    )
    .join(
        meals.select(
            pF.col("meal_id").alias("nonvegan_2"),
            pF.when(pF.col("is_kela"), 1).otherwise(0).alias("kela_nonvegan_2")
        ),
        on='nonvegan_2',
        how='left'
    )
    .filter(pF.col('kela_vegan_1') + pF.col('kela_vegan_2') + pF.col('kela_nonvegan_1') + pF.col('kela_nonvegan_2') >= 2)
    .select(
        "nonvegan_2", "nonvegan_1", "vegan_2", "vegan_1"
    )
    .orderBy(pF.rand())
)

# menus_day.show()

In [10]:
PREFIXES = ["mon", "tue", "wed", "thu", "fri"]

menus_week = None

for prefix in PREFIXES:
    tmp = (
        menus_day
        .sample(0.2, np.random.randint(0, 10000))
        .limit(NUM_COMBOS_PER_DAY)
        .select([pF.col(c).alias(f"{c}_{prefix}") for c in menus_day.columns])
    )

    if menus_week is None:
        menus_week = tmp
    else:
        menus_week = menus_week.crossJoin(tmp)

cols = menus_week.columns
menus_week = menus_week.withColumn("id", pF.monotonically_increasing_id())

menus_id_valid = (
    menus_week
    .melt(ids='id', values=cols, variableColumnName='type', valueColumnName="meal_id")
    .groupBy('id', 'meal_id')
    .count()
    .groupBy('id')
    .agg(pF.max("count").alias("max_per_meal"))
    .filter(pF.col('max_per_meal') <= MAX_MEAL_OCCURENCES)
    .select('id')
)


menus_week = menus_week.join(menus_id_valid, on='id', how='inner')

# menus_week.show(truncate=False)

# Write to Parquet

In [11]:
path_out = f"data/processed/phase_4/menus/{date}"

menus_week.write.parquet(path_out)

In [13]:
df = spark.read.parquet("data/processed/phase_4/menus/2024-12-02")

df.show()

+---+--------------+--------------+-----------+-----------+--------------+--------------+-----------+-----------+--------------+--------------+-----------+-----------+--------------+--------------+-----------+-----------+--------------+--------------+-----------+-----------+
| id|nonvegan_2_mon|nonvegan_1_mon|vegan_2_mon|vegan_1_mon|nonvegan_2_tue|nonvegan_1_tue|vegan_2_tue|vegan_1_tue|nonvegan_2_wed|nonvegan_1_wed|vegan_2_wed|vegan_1_wed|nonvegan_2_thu|nonvegan_1_thu|vegan_2_thu|vegan_1_thu|nonvegan_2_fri|nonvegan_1_fri|vegan_2_fri|vegan_1_fri|
+---+--------------+--------------+-----------+-----------+--------------+--------------+-----------+-----------+--------------+--------------+-----------+-----------+--------------+--------------+-----------+-----------+--------------+--------------+-----------+-----------+
|  2|          6825|          6823|       9026|      14377|       9500131|          6866|    9500146|       8993|          1293|          9094|    9500061|       9047|     

### Polars

In [12]:
# path = "data/processed/phase_4/dim_meals.xlsx"
# dim_meals_raw = pl.read_excel(path).lazy()
# dim_meals_raw.head().collect()


# meals = dim_meals_raw.filter(
#     pl.col('restaurant').is_not_null(),
#     pl.col('restaurant').str.contains(restaurant),
#     pl.col('schoolyear') == pl.lit(schoolyear)
# )

# meals.head().collect()


# meals_vegan = meals.filter(pl.col("meal_type_1") == pl.lit("vegan")).select("meal_id")
# tmp = meals_vegan.with_row_index()
# combo_vegan = (
#     tmp
#     .join(tmp, how='cross')
#     .filter(pl.col("index") < pl.col("index_right"))
#     .select(pl.concat_list("meal_id", "meal_id_right").alias("vegan"))
# )



# meals_notvegan = meals.filter(pl.col("meal_type_1") != pl.lit("vegan")).select("meal_id")
# tmp = meals_notvegan.with_row_index()
# combo_notvegan = (
#     tmp
#     .join(tmp, how='cross')
#     .filter(pl.col("index") < pl.col("index_right"))
#     .select(pl.concat_list("meal_id", "meal_id_right").alias("not_vegan"))
# )

# combo_notvegan = pl.concat([
#     meals_notvegan.select(pl.concat_list("meal_id").alias("not_vegan")),
#     combo_notvegan
# ])



# menus = (
#     combo_vegan
#     .join(combo_notvegan.tail(), how='cross')
#     .select(
#         pl.concat_list("vegan", "not_vegan").list.to_struct()
#     )
#     .unnest('vegan')
#     # .rename(lambda name: name.replace("f", "meal"), strict=False)
#     # .rename({'field_0': 'meal_0', }, strict=False)
# )

# menus.tail().collect()